# Plotting AF3 Results with CSV Logger

This notebook provides examples of how to parse the results of the `CSVLogger` for both training and validation.

In [1]:
# Imports for this notebook
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

In [ ]:
# Path to the log folder
LOG_PATH = Path("/path/to/logs")

# Validation

Workflows to plot and visualize validation metrics

## Plot Results for Most Recent Epoch

In [ ]:
val_df = pd.read_csv(LOG_PATH / "val_metrics/validation_output_all_epochs.csv")

In [ ]:
def plot_validation_results_by_type(
    df: pd.DataFrame,
    ignore_zeros: bool = False,
) -> None:
    """Visualize metrics across all datasets.

    NOTE: Ensure that you first subset the DataFrame to only include the desired epoch.
    
    Args:
        df: Combined DataFrame containing metrics data
        ignore_zeros: Whether to treat zero values as missing data
    """
    # (Copy the DataFrame to avoid modifying the original)
    _df = df.copy()

    # ... subset to only include the desired columns
    _df = _df[["dataset", "by_type_lddt.type", "by_type_lddt.best_of_1_lddt", "by_type_lddt.best_of_5_lddt"]]
    _df = _df.dropna()

    if ignore_zeros:
        _df = _df.replace(0, pd.NA)

    # Prepare data
    melted = pd.melt(_df,
                     id_vars=["dataset", "by_type_lddt.type"],
                     value_vars=["by_type_lddt.best_of_1_lddt", "by_type_lddt.best_of_5_lddt"],
                     var_name='metric',
                     value_name='lddt')
    
    # Create visualization
    sns.set(style="whitegrid", font_scale=1.1)
    plt.figure(figsize=(15, 8))
    
    g = sns.catplot(
        data=melted,
        x='by_type_lddt.type',
        y='lddt',
        hue='metric',
        col='dataset',
        kind='bar',
        estimator='mean',  # Explicitly set to mean aggregation
        ci=None,  # Disable confidence intervals
        height=6,
        aspect=2,
        sharey=False,
        legend_out=False
    )

    # Annotate bars with values
    for ax in g.axes.flat:
        for p in ax.patches:
            ax.annotate(f"{p.get_height():.2f}",
                        (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='center',
                        fontsize=10,
                        color='black',
                        xytext=(0, 7),
                        textcoords='offset points')
        
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        ax.set_xlabel('')
        ax.set_ylabel('LDDT')

    plt.suptitle(f'Model Performance Comparison', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
current_epoch_df = val_df[val_df["epoch"] == val_df["epoch"].max()]
plot_validation_results_by_type(current_epoch_df)

## Plot Validation Curves

In [ ]:
def plot_metric_trend(
    df: pd.DataFrame,
    metric_type: str,
    dataset: str | None = None,
    ignore_zeros: bool = False,
    last_n_epochs: int | None = None
) -> None:
    """Plot best-of-1 vs best-of-5 trends across epochs for a specific metric type.
    
    Args:
        df: Combined metrics DataFrame
        metric_type: The 'type' to plot (e.g., 'protein-ligand')
        dataset_filter: Optional specific dataset to filter
        ignore_zeros: Whether to exclude zero values
        last_n_epochs: Optional number of most recent epochs to plot
    """
    # Filter data
    filtered = df[df['by_type_lddt.type'] == metric_type]
    
    if dataset:
        filtered = filtered[filtered['dataset'] == dataset]
    
    if ignore_zeros:
        filtered = filtered.replace(0, pd.NA).dropna(
            subset=['by_type_lddt.best_of_1_lddt', 'by_type_lddt.best_of_5_lddt']
        )

    if filtered.empty:
        raise ValueError(f"No data found for {metric_type} in dataset {dataset or 'any'}")

    if last_n_epochs:
        max_epoch = filtered['epoch'].max()
        filtered = filtered[filtered['epoch'] > (max_epoch - last_n_epochs)]

    # Aggregate by epoch
    trend_data = filtered.groupby('epoch').agg({
        'by_type_lddt.best_of_1_lddt': 'mean',
        'by_type_lddt.best_of_5_lddt': 'mean'
    }).reset_index()

    # Create plot
    plt.figure(figsize=(12, 6))
    sns.set_style("whitegrid")

    # Plot lines with markers
    sns.lineplot(
        data=trend_data,
        x='epoch',
        y='by_type_lddt.best_of_1_lddt',
        color='#1f77b4',
        label='Best of 1',
        marker='o',
        markersize=8,
        linewidth=2
    )
    
    sns.lineplot(
        data=trend_data,
        x='epoch',
        y='by_type_lddt.best_of_5_lddt',
        color='#ff7f0e',
        label='Best of 5',
        marker='s',
        markersize=8,
        linewidth=2
    )

    # Style plot
    plt.title(f"{metric_type} LDDT Trends\nDataset: {dataset or 'All'}")
    plt.xlabel("Epoch")
    plt.ylabel("Average LDDT")
    plt.legend(title="Strategy")
    plt.grid(alpha=0.3)

    # Add padding to autoscaled y-axis
    plt.ylim(top=min(1.0, plt.ylim()[1] * 1.05))  # Cap at 1.0 if near upper bound
    plt.ylim(bottom=max(0.0, plt.ylim()[0] * 0.95))  # Floor at 0.0 if near lower bound

    # Set x-axis to show only whole numbers
    plt.xticks(ticks=trend_data['epoch'], labels=trend_data['epoch'].astype(int))
    
    # Add final value annotations
    last_values = trend_data.iloc[-1]
    plt.text(
        0.95, 0.15,
        f"Final Bo1: {last_values['by_type_lddt.best_of_1_lddt']:.2f}\nFinal Bo5: {last_values['by_type_lddt.best_of_5_lddt']:.2f}",
        ha='right',
        va='bottom',
        transform=plt.gca().transAxes,
        bbox=dict(facecolor='white', alpha=0.8)
    )

    plt.tight_layout()
    plt.show()

In [ ]:
plot_metric_trend(val_df, 'protein-ligand', dataset="af3_validation", ignore_zeros=False, last_n_epochs=4)

## Visualize Outliers

Identify and visualize outliers

In [ ]:
structures_path = f"{LOG_PATH}/val_structures"

In [ ]:
def get_worst_examples(
    df: pd.DataFrame, 
    metric_type: str, 
    dataset: str = None, 
    epoch: int = None
) -> list:
    """Return example IDs sorted by worst performance for a metric type at specific epoch."""
    filtered = df[df['by_type_lddt.type'] == metric_type]
    
    if dataset:
        filtered = filtered[filtered['dataset'] == dataset]
    
    # Use latest epoch if none specified
    target_epoch = epoch if epoch is not None else filtered['epoch'].max()
    filtered = filtered[filtered['epoch'] == target_epoch]
    
    return (
        filtered[['example_id', 'by_type_lddt.best_of_1_lddt', 'by_type_lddt.best_of_5_lddt']]
        .assign(worst_score=lambda x: x[['by_type_lddt.best_of_1_lddt', 'by_type_lddt.best_of_5_lddt']].min(axis=1))
        .sort_values('worst_score')
        ['example_id']
        .tolist()
    )

In [ ]:
from datahub.common import parse_example_id
from cifutils.utils.visualize import view
from cifutils import parse

dataset = "af3-validation"
metric = "protein-ligand"
latest_epoch = val_df['epoch'].max()

worst_protein_ligand_examples = get_worst_examples(val_df, metric, dataset="af3_validation", epoch=latest_epoch)

# Visualize the worst example
parsed_id = parse_example_id(worst_protein_ligand_examples[0])

# Find the worst example in the structures directory
structure_path_for_epoch = Path(structures_path) / f"epoch_{latest_epoch}" / dataset

if structure_path_for_epoch.exists():
    example_path = next(structure_path_for_epoch.glob(f"*{parsed_id['pdb_id']}_{parsed_id['assembly_id']}*"))

    # ... and visualize
    atom_array = parse(example_path)
    view(atom_array["assemblies"]["1"][0])
else:
    print(f"No structure found for {parsed_id}")

# Training

In [ ]:
csv_path = f"{LOG_PATH}/lightning_logs/version_0/metrics.csv"
_df = pd.read_csv(csv_path)

## Training Curves

In [ ]:
# Subset to the relevant columns
mean_cols = ["train/batch_mean/diffusion_loss", "train/batch_mean/smoothed_lddt_loss", "train/batch_mean/total_loss", "train/batch_mean/distogram_loss"]
per_structure_cols = ["train/per_structure/t", "train/per_structure/diffusion_loss", "train/per_structure/smoothed_lddt_loss"]
train_df = _df[mean_cols + per_structure_cols + ["step", "train/learning_rate"]]

# Remove rows with all NaN values except for the 'step' column
check_cols = [col for col in train_df.columns if col != 'step']
train_df = train_df.dropna(how='all', subset=check_cols)

In [ ]:
def plot_training_metrics(train_df: pd.DataFrame) -> None:
    """Plot all training metrics from a DataFrame."""
    
    processed = (
        train_df
        .groupby('step', as_index=False)
        .mean()
        .melt(id_vars='step', var_name='metric')
    )
    
    # Create visualization
    plt.figure(figsize=(12, 8))
    sns.set_style("whitegrid")
    
    g = sns.FacetGrid(
        processed,
        col='metric',
        col_wrap=3,
        height=4,
        aspect=1.5,
        sharey=False
    )
    
    g.map(sns.lineplot, 'step', 'value', color='#2ca02c')
    g.set_titles("{col_name}")
    g.set_axis_labels("Training Step", "Value")
    
    # Special handling for learning rate
    if 'learning_rate' in processed['metric'].unique():
        g.axes[-1].set_yscale('log')
    
    plt.tight_layout()
    plt.show()

In [ ]:
mean_df = train_df[mean_cols + ["step"]].copy()
mean_df = mean_df.dropna(subset=mean_cols)
plot_training_metrics(train_df[mean_cols + ["step", "train/learning_rate"]])

## Training Loss by T

In [ ]:
def plot_loss_scatter_by_t(
    train_df: pd.DataFrame,
    loss_column: str,
    t_column: str = 'train/per_structure/t',
    n_steps: int = 1000
) -> None:
    """Plot loss values as a scatter plot against train/t values for the most recent N steps with a log-scaled x-axis.
    
    Args:
        train_df: DataFrame containing training metrics
        loss_column: Name of loss column to plot (e.g., 'train/total_loss')
        t_column: Name of the column representing 't' values
        n_steps: Number of recent training steps to analyze (default: 1000)
    """
    train_df = train_df.copy()

    assert loss_column in train_df.columns, f"Loss column '{loss_column}' not found in DataFrame"

    # Get most recent steps
    unique_steps = train_df['step'].dropna().unique()
        
    # Get actual number of available steps
    n_steps = min(n_steps, len(unique_steps))
    latest_steps = np.sort(unique_steps)[-n_steps:]
    
    # Filter recent data
    recent_data = train_df[train_df['step'].isin(latest_steps)]

    # Subset to relevant columns and remove rows with NaN values
    recent_data = recent_data[[t_column, loss_column]]
    
    # Create scatter plot
    plt.figure(figsize=(10, 6))  # Fixed figure size
    sns.set_style("whitegrid")
    
    sns.scatterplot(
        data=recent_data,
        x=t_column,
        y=loss_column,
        color='#2ca02c',
        alpha=0.6
    )
    
    plt.xscale('log')  # Set x-axis to logarithmic scale
    plt.title(f"{loss_column} vs {t_column} (Log Scale)\n(Last {n_steps} Steps)")
    plt.xlabel(t_column)
    plt.ylabel(loss_column)
    
    plt.tight_layout()
    plt.show()


In [ ]:
plot_loss_scatter_by_t(train_df, 'train/per_structure/smoothed_lddt_loss', n_steps=1000)